# 02 · ndarray 핵심 & NumPy → CuPy 포팅

> **CuPy 2일 집중 코스 — Day 1 / 단원 2 (NumPy/SciPy CuPy 프로그래밍)**

이 노트북은 코스의 **레퍼런스 노트북**입니다. NumPy `ndarray`의 구조를 이해하고,
뷰/복사·브로드캐스팅·축 연산을 익힌 뒤, **`import numpy as np` → `import cupy as cp`** 만으로
GPU에서 동작시키는 "드롭인" 포팅을 실습합니다.

### 왜 이 노트북이 '레퍼런스'인가
00에서 GPU가 왜 빠른지, CuPy가 무엇인지 큰 그림을 잡았다면, 02는 **매일 실무에서 실제로 손에 잡히는
도구**를 다룹니다: 배열이 메모리에 어떻게 놓이는지(strides), 언제 복사가 일어나는지(view vs copy),
서로 다른 shape끼리 어떻게 연산되는지(broadcasting), 그리고 CPU/GPU 양쪽에서 동작하는 코드를 어떻게
짜는지(`get_array_module`). 이 네 가지는 03(NumPy 루틴)·04(SciPy 루틴)은 물론 Day 2의 커널 작성까지
**코스 전체에서 계속 재사용되는 어휘**이기 때문에, 이 노트북을 "레퍼런스"라고 부릅니다 — 이후
노트북에서 이해가 막히면 여기로 돌아와 확인하면 됩니다.

### 이 노트북의 흐름
드롭인 대체 확인 → ndarray 내부 구조(anatomy) → 뷰/복사 구분 → 축·집계 → 브로드캐스팅 → 장치
비종속 코드 → 암묵적 전송의 함정 → 멀티 GPU 디바이스 관리 → 포팅 연습. 앞의 다섯 개(2~5절)는
"NumPy를 잘 안다면 이미 아는 내용"을 CuPy 맥락에서 재확인하는 것이고, 뒤의 세 개(6~8절)는
**CuPy에서만 신경 써야 하는 것**입니다 — 이 경계를 의식하며 읽으면 어디에 집중해야 할지 명확해집니다.

## 학습 목표
- `ndarray`의 4요소(**data·dtype·shape·strides**)와 뷰 vs 복사를 설명한다.
- 축(axis) 기반 집계와 **브로드캐스팅**(stretch 규칙)을 능숙하게 쓴다.
- NumPy 코드를 CuPy로 포팅하고, **장치 비종속(agnostic) 코드**를 작성한다.
- **암묵적 전송**과 디바이스 관리의 주의점을 안다.

## 목차
1. [NumPy → CuPy: 드롭인 대체](#1)
2. [ndarray의 구조 (anatomy)](#2)
3. [뷰(View) vs 복사(Copy)](#3)
4. [축(axis)과 집계](#4)
5. [브로드캐스팅: "stretch" 규칙](#5)
6. [장치 비종속 코드 (NumPy dispatch)](#6)
7. [암묵적 전송 주의](#7)
8. [디바이스 관리](#8)
9. [연습문제](#9)
10. [체크포인트](#10)

In [ ]:
import os, sys, time, math
import numpy as np
import cupy as cp
import cupyx as cpx
from course_utils import print_env, bench, gpu_ms, cpu_ms, print_bench, compare
print_env()

<a id="1"></a>
## 1. NumPy → CuPy: 드롭인 대체

CuPy는 **NumPy API를 GPU에서** 구현합니다(백엔드는 CUDA C++). NumPy를 알면 CuPy를 이미 아는 셈입니다.
많은 경우 **import 한 줄**만 바꿔도 GPU에서 돌아갑니다.

### 조금 더 구체적으로: "드롭인"이 실제로 의미하는 것

CuPy의 `cupy.ndarray`는 `numpy.ndarray`와 **동일한 속성 이름**(`.shape`, `.dtype`, `.strides`, `.ndim`
...)과 **동일한 메서드 시그니처**(`.sum(axis=...)`, `.reshape(...)`, `.T` ...)를 갖도록 설계되었습니다.
그래서 함수 안에서 배열 타입을 검사하는 코드만 없다면, `np.xxx`를 `cp.xxx`로 치환하는 것만으로 대부분의
NumPy 코드가 그대로 동작합니다. 내부적으로는 `cp.ones(shape)` 같은 호출이 GPU 메모리 할당(단원 5의
메모리 풀 개념, `05_memory_profiling`에서 다룸)과 CUDA 커널 실행으로 번역됩니다.

다음 절(2절)에서 이 "동일한 API"의 근거가 되는 ndarray의 내부 구조(data·dtype·shape·strides)를 뜯어봅니다.

<img src="images/figures/new_cupy_speedup.png" width="560">

<sub>그림: 동일 코드(배열 생성·연산)를 NumPy(CPU)와 CuPy(GPU)로 실행했을 때의 처리 시간 비교.
막대 높이 차이가 곧 가속 배율이며, 바로 아래 코드 셀에서 `compare()`로 직접 같은 종류의 비교를
재현합니다. 가속 폭은 배열 크기·연산 종류·하드웨어에 따라 달라진다는 점을 01에서 이미
확인했습니다 — 여기서는 ndarray 생성이라는 가장 기본적인 연산에서도 그 차이가 나타남을 보여줍니다.</sub>


In [ ]:
# 같은 (50,500,500) 배열 생성을 CPU vs GPU로 비교 (~95MB)
shape = (50, 500, 500)
compare('ones'+str(shape), lambda: np.ones(shape), lambda: cp.ones(shape), n_repeat=10)

# 타입과 위치 확인
g = cp.ones(shape)
print('type  :', type(g))
print('device:', g.device)

<a id="2"></a>
## 2. ndarray의 구조 (anatomy)

📖 [`cupy.ndarray` 레퍼런스](https://docs.cupy.dev/en/stable/reference/ndarray.html) — CuPy 공식 overview의 **N차원 배열** 구성요소.
지원 dtype: `bool_`, 정수(`int8~64`, `uint8~64`), 실수(`float16/32/64`), 복소수(`complex64/128`). 인덱싱·고급 인덱싱·브로드캐스팅 의미는 `numpy.ndarray`와 동일합니다.

`ndarray`는 파이썬 리스트와 달리 **연속된 고정 크기 메모리 블록**입니다. 네 가지 핵심 속성이 효율의 비결입니다.
- **data**: 원소들이 저장된 메모리 블록에 대한 포인터
- **dtype**: 모든 원소의 균일한 자료형(예: `float32`)
- **shape**: 각 차원의 크기 튜플(예: `(4, 3)`)
- **strides**: 다음 원소로 가기 위한 **바이트 수**(뷰/슬라이싱의 핵심)

### 조금 더 구체적으로: strides는 정확히 무엇인가

파이썬 리스트는 각 원소가 메모리 어디에나 흩어져 있고 포인터로 연결된 구조라, 원소 접근이
간접참조(포인터 따라가기)를 거칩니다. 반면 `ndarray`는 원소들이 **하나의 연속된 블록에 dtype 크기만큼
일정한 간격으로** 촘촘히 박혀 있습니다. 이 덕분에 임의 위치 원소의 주소를 **산술 계산 한 번**으로
바로 구할 수 있고(간접참조 없음), 이것이 NumPy/CuPy가 빠른 근본 이유 중 하나입니다.

`strides`는 각 축에서 인덱스를 1 증가시킬 때 **몇 바이트를 건너뛰어야 하는지**를 담은 튜플입니다.
예를 들어 `float32`(4바이트) 원소로 이루어진 `shape=(3, 4)` 배열이 C order(행 우선)로 저장되어 있다면,
- 마지막 축(열, axis=1)은 원소가 바로 옆에 있으므로 stride는 `4바이트`(dtype 크기 그대로)
- 그 앞 축(행, axis=0)은 한 행이 4개 원소 = `4 × 4바이트 = 16바이트`이므로 stride는 `16바이트`

즉 `strides = (16, 4)`입니다. 일반적인 C order 공식은 `strides[i] = dtype.itemsize × product(shape[i+1:])`
입니다. 원소 `M[i, j]`의 메모리 주소는 `base_address + i*strides[0] + j*strides[1]`로 계산됩니다 —
이 한 줄의 산술식이 사실상 인덱싱의 전부입니다.

이 관점에서 보면 **`reshape`가 왜 공짜인지**도 명확해집니다: 데이터 배치는 그대로 두고 `shape`와
`strides`라는 **메타데이터만** 새로 계산하면 되기 때문입니다(아래 코드 셀에서 `M.strides`로 직접
확인). 전치(`.T`)도 마찬가지로 strides의 순서만 뒤바꿔 표현하는 뷰입니다. 이 아이디어는 바로 다음
3절(뷰 vs 복사)의 핵심 근거가 됩니다.

<img src="images/figures/new_ndarray_anatomy.png" width="620">

<sub>그림: ndarray 객체가 파이썬 레벨에서 노출하는 shape·dtype·strides 등의 메타데이터와, 그 메타데이터가
가리키는 실제 데이터 블록(data pointer) 사이의 관계. 메타데이터를 바꿔치기하면(shape 재해석, stride 재배치)
데이터 복사 없이 다른 "모양"으로 같은 메모리를 볼 수 있다는 것이 이 그림의 핵심 메시지이며, 바로 아래
코드 셀과 3절에서 이 메커니즘을 직접 확인합니다.</sub>


In [ ]:
# 큰 배열로 '밀집 메모리'를 체감 (GPU에 생성)
N = 50_000_000
arr = cp.arange(1, N + 1, dtype=cp.float32)   # 1..N
print('dtype :', arr.dtype)
print('ndim  :', arr.ndim)
print('size  :', f'{arr.size:,}')
print('shape :', arr.shape)
print('nbytes:', f'{arr.nbytes/2**30:.3f} GB')

# reshape는 메타데이터(shape/strides)만 바꾸는 '뷰' (데이터 복사 X)
M = arr.reshape(-1, 5)
print('reshape shape :', M.shape)
print('reshape strides:', M.strides, '(바이트 단위)')

**논리(2D 인덱스) ↔ 물리(1차원 저장) 매핑**: 다차원 인덱스는 결국 1차원 메모리로 펼쳐 저장되며, 그 **순서를 결정하는 것이 strides**입니다.
NumPy/CuPy 기본은 **행 우선(C order)** 입니다. (아래 그림은 매핑 개념을 보여주는 *열 우선* 예시)

<img src="images/figures/new_logical_vs_storage.png" width="660">

<sub>그림: 같은 논리적 2차원 배열이라도, 메모리에 실제로 어떤 순서로 나열되는지는 저장 방식(order)에
달려 있습니다. 그림은 대비를 보여주기 위해 열 우선(Fortran order, `order='F'`) 예시를 사용하지만, 위
코드 셀에서 만든 배열들은 NumPy/CuPy 기본값인 행 우선(C order)입니다 — 같은 논리적 위치라도 order가
다르면 strides 값과 물리적 저장 순서가 달라진다는 점만 그림에서 읽어내면 충분합니다.</sub>

### 조금 더 구체적으로: order가 성능에 미치는 영향

C order 배열에서 **마지막 축을 따라가며(행 내부에서 열 방향으로) 순회**하면 메모리 주소가 연속적으로
증가하므로 캐시(CPU)나 메모리 코얼레싱(coalescing, GPU)에 유리합니다. 반대로 **첫 축을 따라 순회**하면
매번 큰 보폭(stride)으로 건너뛰어야 해 접근 지역성이 떨어집니다. CuPy에서는 이 코얼레싱 문제가 GPU
워프(warp) 단위 메모리 접근 효율에 직결되므로, 큰 배열을 다룰 때는 어떤 축으로 반복·연산하는지가
실제 처리 속도에 영향을 줄 수 있습니다 — 세부 최적화는 Day 2 커널 작성(단원 7~11)에서 다시 다룹니다.

<a id="3"></a>
## 3. 뷰(View) vs 복사(Copy)

슬라이싱·`reshape`·전치(`.T`)는 보통 **뷰**를 반환합니다 — 메타데이터만 바뀌고 물리 메모리는 공유하므로 거의 즉시.
반면 `A + B`, `A * 5` 같은 연산은 **새 배열(복사)** 를 할당합니다. 순차 연산이 많으면 복사가 성능 함정이 됩니다.
아래 슬라이싱 도해에서 노란 테두리가 "원본의 뷰"입니다.

### 조금 더 구체적으로: 언제 뷰이고 언제 복사인가

경험적 구분법은 다음과 같습니다.

| 연산 | 결과 | 이유 |
|------|------|------|
| 기본 슬라이싱 `a[1:3]`, `a[:, 0]` | **뷰** | 시작점·간격(stride)만 알면 원본 데이터를 그대로 가리킬 수 있음 |
| `a.reshape(...)` (연속 메모리인 경우) | **뷰** | shape·strides 재계산만으로 표현 가능 |
| `a.T`, `a.transpose()` | **뷰** | 축 순서만 바뀌므로 strides 순서만 재배열 |
| 고급 인덱싱 `a[[0,2,4]]`, `a[mask]` | **복사** | 결과가 원본에서 연속적이지 않아 하나의 (offset, stride) 조합으로 표현 불가 |
| 산술 연산 `a + b`, `a * 5` | **복사** | 결과를 저장할 새 메모리가 필요 |
| `a.copy()` | **복사(명시적)** | 사용자가 의도적으로 메모리를 분리 |

뷰는 `.base` 속성(뷰라면 원본 배열을, 원본이거나 복사본이면 `None`을 가리킴)이나, 코드 셀처럼
`.data.ptr`(실제 메모리 시작 주소)이 원본과 같은지 비교해 확인할 수 있습니다. **뷰의 위험성**은
바로 이 메모리 공유에서 나옵니다 — 뷰를 통해 값을 바꾸면 원본도 함께 바뀌므로, 의도치 않게
원본을 오염시키는 부작용(side effect)이 생길 수 있습니다. 이 함정은 9절 연습문제 B(뷰로 원본 수정)에서
직접 다룹니다.

<img src="images/figures/new_views_slicing.png" width="680">

<sub>그림: 원본 배열에 슬라이싱을 적용했을 때(노란 테두리) 새 메모리를 만들지 않고 원본 데이터의
일부를 가리키는 "주소표"만 새로 생성되는 모습. 뷰의 shape는 슬라이스한 범위에 맞춰 작아지지만,
내부 포인터는 원본 데이터 블록 안의 한 지점을 계속 가리킵니다 — 이 때문에 뷰를 통해 값을 쓰면
원본도 함께 바뀝니다. 바로 아래 코드 셀에서 `.data.ptr` 비교로 이를 직접 검증합니다.</sub>


In [ ]:
a = cp.arange(12, dtype=cp.float32).reshape(3, 4)
v = a[:, :2]            # 슬라이싱 -> 뷰
c = a[:, :2].copy()     # 명시적 복사
r = a.reshape(2, 6)     # reshape -> 뷰
print('슬라이싱 뷰? (메모리 공유):', v.data.ptr == a.data.ptr)
print('복사본?    (메모리 분리):', c.data.ptr == a.data.ptr)
print('reshape 뷰?            :', r.data.ptr == a.data.ptr)

**순차 연산의 복사 함정**: 아래 `seq`의 각 줄(`x*5`, `x*x`, `x+x`)은 매번 새 배열을 만듭니다.
그래도 GPU에 머무는 한 host 전송은 없고, CuPy는 필요할 때만 GPU→CPU로 가져옵니다.

### 조금 더 구체적으로: 복사가 왜 느린가

`x = x * 5`처럼 **비-제자리(out-of-place)** 연산은 매번 (1) 결과를 담을 새 GPU 메모리를 할당하고,
(2) 그 메모리에 결과를 써넣습니다. `(50, 500, 500)` `float32` 배열 하나가 약 `50×500×500×4바이트 ≈ 47.7 MB`이므로,
`seq()`처럼 세 줄을 거치면 그때마다 원본을 읽고 결과를 새로 쓰는 대역폭 소비가 발생하고, 옛 배열은
파이썬 참조가 사라지는 즉시 GPU 메모리 풀로 반환됩니다(메모리 풀 재사용 메커니즘은
`05_memory_profiling`에서 다룹니다). 반면 `x *= 5`처럼 **제자리(in-place, augmented assignment)** 연산은
새 메모리를 할당하지 않고 기존 버퍼에 바로 덮어씁니다 — 할당 오버헤드가 사라지고, 큰 배열일수록
그 차이가 뚜렷해집니다. 아래 두 코드 셀(`seq` vs `seq_efficient`)이 이 차이를 실측합니다.

> ⚠️ **제자리 연산의 주의점**: `x *= 5`는 `x`가 가리키는 메모리를 바로 덮어쓰므로, 그 메모리를
> 다른 변수(뷰)가 함께 보고 있다면 **그 변수도 값이 바뀝니다.** "복사를 줄이는 것"과 "원본을 실수로
> 바꾸는 것"은 종이 한 장 차이이니, 함수에 넘어온 배열을 제자리로 바꿀 때는 호출자가 그 부작용을
> 알고 있는지 항상 확인하세요.

In [ ]:
# GPU에서 복사를 일으키는 코드. 매번 새로운 배열을 만들어서 GPU 메모리를 낭비함.
def seq(x):
    x = x * 5
    x = x * x
    x = x + x
    return x

x_gpu = cp.ones((50, 500, 500), dtype=cp.float32)
print_bench(bench(lambda: seq(x_gpu), n_repeat=10, name='sequential(copies)'))

In [ ]:
# 복사 함정을 해결한 올바른 코드
def seq_efficient(x) :
    x *= 5   # 새로운 배열을 만들지 않고, 원래 x가 있던 자리에 5를 곱해버림!
    x *= x   # 원래 자리에 그대로 제곱을 해버림!
    x += x   # 원래 자리에 그대로 더해버림!
    return x

x_gpu = cp.ones((50, 500, 500), dtype=cp.float32)
print_bench(bench(lambda: seq(x_gpu), n_repeat=10, name='sequential(copies)'))
print_bench(bench(lambda: seq_efficient(x_gpu), n_repeat=10, name='sequential(seq_efficient)'))

<a id="4"></a>
## 4. 축(axis)과 집계

`sum/mean/max` 등 집계는 **접을(reduce) 축**을 지정합니다.
- `axis=0`: 첫 번째 차원(행)을 접음 → **열별** 결과
- `axis=1`: 두 번째 차원(열)을 접음 → **행별** 결과
- `keepdims=True`: 차원 수를 보존(브로드캐스팅에 유용)

### 조금 더 구체적으로: "접는다"는 것이 헷갈릴 때

`axis=k`를 지정하면 결과 shape에서 **k번째 차원이 사라집니다**(`keepdims=False`일 때). 예를 들어
`shape=(3, 4)`인 행렬에서 `axis=0`으로 합하면 결과 shape는 `(4,)`가 되고, `axis=1`로 합하면 `(3,)`이 됩니다.
"axis=0을 지정했는데 왜 열별 결과가 나오지?"라는 흔한 혼동은, **"그 축 방향을 따라가며 값을 합친다"**로
바꿔 생각하면 풀립니다 — axis=0을 따라간다는 것은 행 인덱스를 훑는다는 뜻이므로, 각 열마다 여러 행의
값을 하나로 뭉치는 것입니다. 아래 그림에서 화살표 방향이 바로 이 "따라가는 축"을 보여줍니다.

`keepdims=True`는 차원 수를 유지해 `(3,4)`를 `axis=1`로 합칠 때 `(3,)`이 아니라 `(3,1)`을 반환합니다.
당장은 불필요해 보이지만, 5절 브로드캐스팅에서 원본과 다시 연산(예: 행별 정규화)할 때 shape가
자동으로 맞아떨어지게 하는 핵심 장치입니다 — `(3,1)`은 `(3,4)`와 브로드캐스팅 가능하지만 `(3,)`은
때에 따라 의도와 다르게 해석될 수 있습니다.

<img src="images/figures/new_array_functions_axis.png" width="640">

<sub>그림: 2차원 배열에서 axis=0(첫 번째 차원을 따라, 열별 결과)과 axis=1(두 번째 차원을 따라,
행별 결과)로 집계할 때 어떤 방향으로 값이 합쳐지는지를 화살표로 보여줍니다. 화살표가 가리키는
방향이 "사라지는(reduce되는)" 축이라고 기억하면 axis 번호와 결과 방향을 헷갈리지 않습니다.</sub>


In [ ]:
M = cp.arange(1, 13, dtype=cp.float32).reshape(3, 4)
print('M =\n', cp.asnumpy(M))
print('axis=0 (열별 합):', cp.asnumpy(M.sum(axis=0)))
print('axis=1 (행별 합):', cp.asnumpy(M.sum(axis=1)))
print('keepdims shape :', M.sum(axis=1, keepdims=True).shape)

<a id="5"></a>
## 5. 브로드캐스팅: "stretch" 규칙

서로 다른 모양의 배열 간 연산 시, 작은 쪽을 "늘려(stretch)" 맞춥니다. **호환 규칙**: 두 차원이
(1) 같거나 (2) 한쪽이 1이면 호환. 1인 차원은 **메모리 복사 없이** 논리적으로 확장됩니다.

### 조금 더 구체적으로: 규칙을 적용하는 절차

두 배열의 shape를 비교할 때는 **차원 수가 다르면 짧은 쪽 앞에 1을 채워** 맞춘 뒤, **뒤에서부터
(마지막 축부터) 차례로** 비교합니다. 예를 들어 `(4, 5)`와 `(5,)`를 더하면, 짧은 쪽 앞에 1을 채워
`(1, 5)`로 맞추고 뒤에서부터 비교합니다: 마지막 축 `5 == 5`(같음, 호환), 그 앞 축 `4` vs `1`(한쪽이
1이므로 호환) → 결과 shape는 `(4, 5)`. 이때 `(5,)` 쪽은 실제로 메모리에 5개 값이 4번 복제되어
저장되는 것이 **아니라**, 연산 커널이 그 축의 인덱스를 매번 0으로 취급(stride 0으로 순회)해
"논리적으로만" 늘어난 것처럼 동작합니다 — 그래서 브로드캐스팅은 메모리를 추가로 쓰지 않습니다.

두 차원이 **같지도 않고 둘 다 1도 아니면** 호환되지 않아 `ValueError`가 발생합니다(아래 코드 셀의
`(3,2) + (3,)` 예시 — 마지막 축끼리 비교하면 `2` vs `3`으로 둘 다 1이 아니라서 실패). 아래 두 그림이
각각 성공(ok)·실패(err) 케이스의 shape 정렬을 시각적으로 보여줍니다.

<img src="images/figures/new_broadcasting_ok.png" width="520">

<sub>그림: 호환되는 shape 쌍의 예 — 더 작은 배열(주로 크기 1인 차원)이 물리적 복사 없이
논리적으로 "늘어나"(stretch) 큰 배열과 같은 shape로 맞춰지는 모습.</sub>


<img src="images/figures/new_broadcasting_err.png" width="520">

<sub>그림: 호환되지 않는 shape 쌍의 예 — 마지막 축부터 비교했을 때 두 차원이 같지도 않고 둘 다 1도
아닌 경우로, 이런 조합은 `ValueError: operands could not be broadcast together`로 즉시 실패합니다.
아래 코드 셀에서 이 에러를 직접 발생시켜 확인합니다.</sub>


In [ ]:
# 행별 정규화: (4,5)를 행 합 (4,1)로 나누면 열 방향으로 stretch
M = cp.random.random((4, 5), dtype=cp.float32)
row_sums = M.sum(axis=1, keepdims=True)     # (4,1)
Mn = M / row_sums                            # 브로드캐스팅
print('정규화 후 각 행 합:', cp.asnumpy(Mn.sum(axis=1)))   # ~1.0

# 호환되지 않는 경우
try:
    _ = cp.ones((3, 2)) + cp.arange(3)
except ValueError as e:
    print('ValueError:', e)

<a id="6"></a>
## 6. 장치 비종속 코드 (NumPy dispatch)

`cp.get_array_module(x)` 는 입력이 NumPy면 `numpy`, CuPy면 `cupy` 모듈을 돌려줍니다.
이를 쓰면 **CPU/GPU 양쪽에서 동작하는 함수** 하나를 작성할 수 있습니다.
또한 많은 `np.*` 함수는 `__array_function__`(NEP 18) 덕분에 **CuPy 배열을 넣으면 자동으로 GPU에서 실행**됩니다.

### 조금 더 구체적으로: 왜 장치 비종속 코드가 실무에서 중요한가

연구·상용 코드베이스는 보통 "GPU가 있으면 GPU, 없으면 CPU로 폴백"하는 형태를 요구합니다.
`if isinstance(x, cp.ndarray): ...` 같은 분기를 함수마다 넣는 대신, `xp = cp.get_array_module(x)`
한 줄로 시작해 이후 코드에서 `np.`/`cp.` 대신 **`xp.`** 를 쓰면 함수 본문을 전혀 손대지 않고도
양쪽에서 동작합니다. 이 패턴은 아래 6절 코드 셀의 `standardize`, 그리고 9절 연습문제의 `transform`,
`col_minmax`에서 반복해 사용되며, 이 코스 이후에도 CuPy 기반 라이브러리(예: cuML, cuDF 생태계)
전반에서 흔히 볼 수 있는 관용구입니다.

NEP 18(`__array_function__` protocol)은 NumPy 함수가 **자신에게 익숙하지 않은 타입의 입력**을 받았을 때,
그 타입이 직접 처리 방법을 제공한다면 위임(dispatch)하도록 하는 NumPy의 공식 확장 메커니즘입니다.
CuPy가 이 프로토콜을 구현해두었기 때문에, `cp.linalg.svd` 대신 **`np.linalg.svd`를 그대로 호출**해도
입력이 CuPy 배열이면 자동으로 CuPy(→ cuSOLVER)로 위임되어 GPU에서 실행됩니다. 다음 셀의 그림이
이 위임 흐름을 단계별로 보여줍니다.

### NumPy ↔ CuPy 자동 디스패치 흐름 (`__array_function__` protocol)

아래는 `np.linalg.svd(x_gpu)` 호출 시 내부적으로 일어나는 위임 과정을 단계별로 나눈 것입니다.

1. **사용자 코드**
   `np.linalg.svd(x_gpu)` 호출

2. **NumPy 내부**
   입력이 CuPy 배열임을 감지(`__array_function__` protocol) → 자체 연산 포기. Cupy 구현으로 위임(dispatch)

3. **CuPy 내부**
   "바톤 터치!" → `cp.linalg.svd(x_gpu)` 호출로 위임

4. **GPU 하드웨어**
   CuPy가 준비한 CUDA 커널(cuSOLVER 라이브러리) 실행

> 핵심: NumPy 함수를 그대로 호출해도, 입력이 CuPy 배열이면 NumPy가 알아서 CuPy 구현으로 위임(dispatch)합니다.

참고로 `get_array_module`(위 6절)은 **사용자가 명시적으로** 어느 모듈을 쓸지 고르는 방식이고,
`__array_function__`은 NumPy가 **내부적으로 자동** 위임하는 방식입니다 — 목적(장치 비종속성)은
같지만 메커니즘이 다르다는 점을 기억해두면, "왜 `np.`를 그대로 썼는데 GPU에서 도나?"라는 의문이
풀립니다.

In [ ]:
def standardize(x):
    # 1. 입력 x가 NumPy 배열이면 np를, CuPy 배열이면 cp를 반환합니다.
    xp = cp.get_array_module(x)     
    mean = xp.mean(x)
    std = xp.std(x)    
    return (x - mean) / (std + 1e-8)

a_np = np.random.randn(100_000).astype(np.float32)
print('NumPy 입력 -> 결과 type:', type(standardize(a_np)))
print('CuPy  입력 -> 결과 type:', type(standardize(cp.asarray(a_np))))

# np.* 함수가 CuPy 배열로 디스패치되어 GPU에서 실행되는 예 (SVD, O(N^3))
x_gpu = cp.random.random((1500, 600), dtype=cp.float32)
print_bench(bench(lambda: np.linalg.svd(x_gpu, compute_uv=False), n_repeat=3, name='np.linalg.svd→GPU'))

<a id="7"></a>
## 7. 암묵적 전송 주의

숨은 성능 함정은 **CPU↔GPU 암묵적 전송**입니다. CuPy는 일부를 막아줍니다:
`np.asarray(gpu_array)`처럼 `__array__`로 host 변환을 시도하면 **조용히 복사하지 않고 `TypeError`** 를 냅니다.
하지만 **출력(print), 스칼라 변환(`float`, `.item()`)** 등은 암묵적으로 GPU→CPU 전송을 일으킵니다.

### 조금 더 구체적으로: 왜 이런 비대칭적 정책인가

CuPy 개발진은 "**명시적 전송은 허용, 암묵적 대량 복사는 차단**"이라는 원칙을 따릅니다.
`np.asarray(gpu_array)`를 그대로 허용하면, 사용자가 무심코 `np.` 함수에 CuPy 배열을 넘겼을 때
NumPy가 배열 전체를 **아무 경고 없이** host로 복사해버릴 위험이 있습니다 — 이는 00에서 강조한
"전송 비용은 크다"는 원칙에 정면으로 위배되는 조용한 성능 함정이므로, CuPy는 이를 아예 `TypeError`로
막아 사용자가 `cp.asnumpy()`를 **의식적으로** 호출하도록 강제합니다.

반면 `float(x)`, `x.item()`, `print(x)`처럼 **배열 하나(또는 스칼라)를 파이썬 값으로 뽑아내는 연산**은
차단하지 않습니다. 이런 연산은 대개 결과 확인·로깅처럼 "사용자가 의도한" 마지막 단계이고, 데이터
크기도 작아 전송 비용이 크지 않기 때문입니다. 다만 **반복문 안에서** `float(x)`를 매번 호출하면
00의 7절에서 본 것과 동일한 전송 병목이 누적되므로, 이 역시 "가끔 vs 매번"의 문제입니다.

> ⚠️ CuPy가 막아주지 못하는 암묵적 전송도 있습니다. 예를 들어 `if cupy_array > 0:` 같은 조건문은
> 배열을 스칼라로 강제 변환하려다 에러가 나거나(다차원인 경우), 단일 원소 배열이면 조용히 값을
> 꺼내 GPU→CPU 동기화를 유발합니다. "값을 하나라도 host로 꺼내는 연산인가?"를 항상 의식하는 것이
> 가장 확실한 방어책입니다.

In [ ]:
g = cp.arange(5, dtype=cp.float32)
try:
    np.asarray(g)            # 암묵적 host 복사 시도 -> 차단
except TypeError as e:
    print('np.asarray(gpu) 차단:', str(e)[:90])

print('명시적 전송 asnumpy :', cp.asnumpy(g))   # 의도적 전송 (OK)
print('스칼라 변환 float()  :', float(g.sum()))   # 암묵적 전송 발생

<a id="8"></a>
## 8. 디바이스 관리

GPU가 여러 개면 `with cp.cuda.Device(i):` 로 특정 장치에 배열을 만들 수 있습니다.
CuPy 연산은 보통 입력들이 **같은 장치**에 있어야 합니다.

### 조금 더 구체적으로: 디바이스 컨텍스트와 흔한 실수

CuPy는 "현재 활성 디바이스(current device)"라는 개념을 스레드 단위로 유지합니다. `cp.cuda.Device(i)`를
`with` 블록으로 열면 그 블록 안에서 생성되는 모든 배열·커널 실행이 디바이스 `i`를 대상으로 하고,
블록을 벗어나면 이전 디바이스로 자동 복귀합니다(파이썬 컨텍스트 매니저 관용구와 동일). 지정하지
않으면 기본값은 디바이스 0번입니다.

멀티 GPU 환경에서 가장 흔한 실수는 **서로 다른 디바이스에 있는 배열끼리 연산하려는 것**입니다 —
예를 들어 디바이스 0에서 만든 배열과 디바이스 1에서 만든 배열을 그대로 더하면 CuPy는 두 메모리가
물리적으로 다른 GPU에 있어 직접 접근할 수 없으므로 에러를 냅니다. 배열을 다른 디바이스로 옮기려면
`cp.asarray(x, ...)`나 명시적 복사가 필요하며(멀티 GPU 간 통신은 NVLink가 있으면 PCIe보다 훨씬
빠릅니다), 진짜 멀티 GPU 병렬화(여러 GPU에 작업을 나눠 동시 실행)는 이 코스 범위를 넘어서는
고급 주제입니다 — 여기서는 "GPU를 선택하는 문법"과 "다른 디바이스 배열은 섞이지 않는다"는 사실만
확실히 짚고 넘어갑니다.

In [ ]:
print('현재 device id:', cp.cuda.Device().id)
with cp.cuda.Device(0):
    x = cp.random.random((1000, 1000), dtype=cp.float32)
print('x.device:', x.device)

<a id="9"></a>
## 9. 연습문제 — 장치 비종속 변환 포팅

입력 배열 `x`에 대해 다음을 수행하는 **장치 비종속** 함수 `transform(x)` 를 완성하세요.
1) z-score 표준화 `z = (x - mean) / (std + 1e-8)`
2) `z`를 `(-1, 5)`로 reshape
3) 각 행을 그 행의 **절댓값 최댓값**으로 나눠 반환

조건: `cp.get_array_module`을 써서 NumPy/CuPy 모두에서 동작하게 하고, 같은 입력에 대해 결과가 일치함을 확인하세요.

이 연습은 지금까지 배운 요소를 한 함수에 모두 조합합니다: (1) 6절의 `xp = get_array_module` 패턴,
(2) 2절에서 본 `reshape`는 뷰이므로 추가 복사 비용이 거의 없다는 사실, (3) 4절의 `axis`/`keepdims`를
이용한 행별 집계, (4) 5절의 브로드캐스팅으로 `(-1, 5)` 배열을 행별 최댓값 `(-1, 1)`로 나누는 연산.
막히면 각 절로 돌아가 해당 개념만 다시 확인하세요.

In [ ]:
def transform(x):
    # TODO: xp = cp.get_array_module(x) 를 사용해 위 1)~3)을 구현
    raise NotImplementedError

# 검증 (구현 후 주석 해제): 같은 host 입력 -> CPU/GPU 결과 일치
x_np = np.random.randn(2_000_000).astype(np.float32)
# ref = transform(x_np)
# out = cp.asnumpy(transform(cp.asarray(x_np)))
# np.testing.assert_allclose(ref, out, rtol=1e-4, atol=1e-4); print('정확성 OK')
# compare('transform', lambda: transform(x_np), lambda: transform(cp.asarray(x_np)), n_repeat=5)

<details>
<summary>💡 해답 보기</summary>

```python
def transform(x):
    xp = cp.get_array_module(x)
    z = (x - xp.mean(x)) / (xp.std(x) + 1e-8)
    M = z.reshape(-1, 5)
    return M / xp.abs(M).max(axis=1, keepdims=True)

x_np = np.random.randn(2_000_000).astype(np.float32)
ref = transform(x_np)
out = cp.asnumpy(transform(cp.asarray(x_np)))
np.testing.assert_allclose(ref, out, rtol=1e-4, atol=1e-4)
print('정확성 OK')
compare('transform', lambda: transform(x_np),
        lambda: transform(cp.asarray(x_np)), n_repeat=5)
```

포인트: `xp = cp.get_array_module(x)` 하나로 NumPy/CuPy 양쪽을 지원합니다. 함수 본문은 `np.`/`cp.`를
전혀 언급하지 않고 오직 `xp.`만 사용하므로, 호출하는 쪽이 NumPy 배열을 넘기든 CuPy 배열을 넘기든
**같은 코드가 그대로 재사용**됩니다. `reshape(-1, 5)`는 데이터 복사 없는 뷰이고, `xp.abs(M).max(axis=1, keepdims=True)`는
4절의 축 집계와 5절의 브로드캐스팅(`(-1,5)` ÷ `(-1,1)`)을 그대로 응용한 것입니다.
같은 host 입력을 넣으면 결과가 일치하지만, 각자 난수를 생성하면 RNG가 달라 일치하지 않습니다(단원 1 참고).
`compare()` 호출로 같은 연산의 CPU/GPU 처리 시간까지 함께 확인할 수 있습니다 — 이 정도로 작은 배열(2M
원소)에서는 GPU가 오히려 손해일 수도 있다는 점을 01의 손익분기점 논의와 함께 떠올려 보세요.
</details>

## 🧪 추가 연습

**연습 A — 열별 Min-Max 정규화** (브로드캐스팅): 각 열을 `[0,1]`로 정규화하는 장치 비종속 함수를 완성하세요.

행별 정규화(5절 코드 셀에서 `axis=1, keepdims=True`로 이미 다룸)와 짝을 이루는 연습입니다. 이번에는
`axis=0`으로 열 방향을 접어 `(1, 열개수)` shape의 min/max를 얻고, 이를 `(행개수, 열개수)` 원본과
브로드캐스팅으로 나누게 됩니다 — 5절의 stretch 규칙이 "1인 차원"을 어느 축에 두느냐만 바꿔 그대로
재사용되는 좋은 예입니다.

In [ ]:
def col_minmax(x):
    # TODO: 열별 min/max로 (x-min)/(max-min) 정규화 (cp.get_array_module 사용)
    raise NotImplementedError

X_np = np.random.randn(1000, 8).astype(np.float32)
# ref = col_minmax(X_np); out = cp.asnumpy(col_minmax(cp.asarray(X_np)))
# np.testing.assert_allclose(ref, out, rtol=1e-4, atol=1e-4); print('OK')

<details><summary>💡 해답 보기</summary>

```python
def col_minmax(x):
    xp = cp.get_array_module(x)
    mn = xp.min(x, axis=0, keepdims=True)
    mx = xp.max(x, axis=0, keepdims=True)
    return (x - mn) / (mx - mn + 1e-8)

X_np = np.random.randn(1000, 8).astype(np.float32)
ref = col_minmax(X_np)
out = cp.asnumpy(col_minmax(cp.asarray(X_np)))
np.testing.assert_allclose(ref, out, rtol=1e-4, atol=1e-4); print('OK')
```

포인트: `axis=0, keepdims=True`로 얻은 `mn`/`mx`는 shape `(1, 8)`이 되어, 원본 `(1000, 8)`과
브로드캐스팅됩니다 — 행 방향(1000개)이 stretch되어 각 열마다 동일한 min/max가 모든 행에 적용됩니다.
`+ 1e-8`은 어떤 열이 상수값(min == max)이라 나누기 0이 되는 경우를 방지하는 안전장치입니다.
</details>

**연습 B — 뷰로 원본 수정**: 슬라이싱 뷰에 대입해 **원본이 바뀌는지** 확인하고, `.copy()`와 대비하세요.
- 슬라이싱 (Slicing): 덩어리에서 일부를 잘라내는 행위. 파이썬에서 a[:2, :2]라고 쓰는 것이 바로 슬라이싱임.
- 뷰 (View): 새로 만든 데이터가 아니라, 원본을 바라보는 주소표 (뷰). 슬라이싱을 했을 때 컴퓨터는 데이터를 새로 복사하지 않고, "원본의 어느 주소부터 어느 주소까지 바라보면 된다"라는 메모리 주소표(View)만 새로 생성함.

이 연습은 3절에서 다룬 "뷰는 원본과 메모리를 공유한다"는 사실이 **부작용(side effect)** 으로
이어질 수 있음을 직접 체험하는 것입니다. 실무에서는 함수에 배열을 넘겼을 때 그 함수가 슬라이싱한
뷰를 수정하면 호출자의 원본까지 조용히 바뀌는 버그로 이어지기 쉽습니다 — 원본을 보존해야 한다면
`.copy()`로 명시적으로 메모리를 분리해야 합니다.

In [ ]:
def zero_block(a):
    # TODO: a의 좌상단 2x2를 '뷰'로 0으로 만들어 원본도 바뀌게 하세요 (복사 금지)
    raise NotImplementedError

A = cp.arange(16, dtype=cp.float32).reshape(4, 4)
# print(zero_block(A))

<details><summary>💡 해답 보기</summary>

```python
def zero_block(a):
    v = a[:2, :2]     # 슬라이싱 -> 뷰 (메모리 공유)
    v[:] = 0          # 뷰에 대입하면 원본도 변경
    return a

A = cp.arange(16, dtype=cp.float32).reshape(4, 4)
print(zero_block(A))   # 좌상단 2x2가 0
# 대조: B[:2,:2].copy() 에 대입하면 원본은 그대로
```

포인트: `v[:] = 0`처럼 슬라이스 객체 전체에 **제자리로 대입**하면(`v = 0`이 아니라 `v[:] = 0`임에 주의)
`v`가 가리키는 원본 메모리 위치에 직접 0이 써집니다. 만약 `v = a[:2, :2].copy()`로 만들었다면 `v`는
독립된 메모리를 가리키므로 `v[:] = 0`을 해도 `a`는 전혀 바뀌지 않습니다 — "뷰냐 복사냐"가 결과를
완전히 다르게 만드는 가장 직접적인 예시입니다.
</details>

<a id="10"></a>
## 10. 체크포인트

- [ ] ndarray의 data·dtype·shape·strides를 설명할 수 있다
- [ ] 슬라이싱/`reshape`이 뷰인지 복사인지 `data.ptr`로 확인했다
- [ ] `axis`와 `keepdims`로 원하는 집계를 만들 수 있다
- [ ] 브로드캐스팅 호환 규칙(같거나 1)을 적용해 행별 정규화를 했다
- [ ] `cp.get_array_module`로 장치 비종속 함수를 작성했다
- [ ] `np.asarray(gpu)`가 차단되는 이유와 암묵적 전송을 안다

다음: **`03_numpy_routines`** — NumPy 루틴(`cupy.*` 모듈 함수·`linalg`·`fft`·`random`)을 예제로 다룹니다.